# CivicBook Component 4: translation for the language confound

**Problem.** In the training data every LECO complaint is English and every MC complaint is Sinhala, so the model learned *language → institution*.
**Fix.** Translate part of the **training and validation** complaints so that each institution appears in both languages:

| Direction | From | To |
|---|---|---|
| LECO | English | Sinhala |
| MC | Sinhala | English |

Model: **NLLB-200** (Meta AI, `facebook/nllb-200-distilled-1.3B`, licence CC-BY-NC 4.0, research use). It runs inside this notebook, so the complaint text never leaves Kaggle.

**Notebook settings** (right panel): Accelerator **GPU T4 x1** · Internet **On** · Input: your **private** dataset with `source_split.xlsx` (from `build_dataset.py`).

**Privacy.** Keep this notebook and its dataset **Private**. The notebook does not print complaint text; read the results in `translations.xlsx` on your own computer.

**Test complaints are never translated** (they stay unseen, so there is no leakage).

| Part | What it does |
|---|---|
| 1 | Settings |
| 2 | Load `source_split.xlsx` and choose the rows to translate |
| 3 | Text helpers: sentence splitting, protecting `<NUMBER>` / `<EMAIL>`, quality flags |
| 4 | Load NLLB and test it on two invented sentences |
| 5 | Translate |
| 6 | Write `translations.xlsx` with a review sample |

## Part 1: Settings

In [ ]:
from pathlib import Path

MODEL_NAME = "facebook/nllb-200-distilled-1.3B"   # smaller/faster: facebook/nllb-200-distilled-600M
SEED = 42

LECO_TO_SINHALA_SHARE = 0.35      # share of English LECO complaints translated to Sinhala
MC_TO_ENGLISH_SHARE = 0.50        # share of Sinhala MC complaints translated to English
SPLITS_TO_TRANSLATE = ["train", "validation"]    # NEVER add "test"
MAX_WORDS = 150                   # longer complaints are not used by build_dataset anyway

BATCH_SIZE = 16                   # lower to 8 if you get "CUDA out of memory"
NUM_BEAMS = 4
MAX_SENTENCE_WORDS = 60           # longer sentences are cut at commas before translating
REVIEW_SAMPLE_PER_DIRECTION = 50  # rows you will check by hand

QUICK_TEST = False                # True = only 20 rows per direction (check everything works first)

OUT = Path("/kaggle/working")
assert "test" not in SPLITS_TO_TRANSLATE, "Test complaints must never be translated (leakage)."

## Part 2: Load the source complaints and choose rows to translate

In [ ]:
import pandas as pd

found = sorted(Path("/kaggle/input").rglob("source_split.xlsx"))
if not found:
    raise FileNotFoundError("source_split.xlsx not found. Add your private dataset with this file "
                            "(Add Input, then Your Datasets).")
src = pd.read_excel(found[0])
print("Loaded", found[0], "-", len(src), "real complaints")

need = {"source_id", "institution", "department", "text", "split", "language", "n_words"}
missing = need - set(src.columns)
assert not missing, f"source_split.xlsx is missing columns: {missing}"

if "translated" in src.columns:                  # never translate a translation
    src = src[~src["translated"].astype(bool)]
pool = src[src["split"].isin(SPLITS_TO_TRANSLATE) & (src["n_words"] <= MAX_WORDS)]

def take(df, share):
    # same share from every department and split (stratified), reproducible
    return (df.groupby(["department", "split"], group_keys=False)
              .sample(frac=share, random_state=SEED))

leco = take(pool[(pool.institution == "LECO") & (pool.language == "en")], LECO_TO_SINHALA_SHARE)
mc = take(pool[(pool.institution == "MC") & (pool.language == "si")], MC_TO_ENGLISH_SHARE)
if QUICK_TEST:
    leco, mc = leco.head(20), mc.head(20)

jobs = [  # (name, rows, NLLB source code, NLLB target code, target language)
    ("LECO en->si", leco, "eng_Latn", "sin_Sinh", "si"),
    ("MC si->en", mc, "sin_Sinh", "eng_Latn", "en"),
]
for name, rows, *_ in jobs:
    print(f"\n{name}: {len(rows)} complaints")
    print(rows.groupby(["department", "split"]).size().unstack(fill_value=0).to_string())

## Part 3: Text helpers
* **Sentences.** NLLB is trained on single sentences; long inputs can lose or repeat parts. Each complaint is split into sentences, translated, and joined again.
* **Placeholders.** `<NUMBER>` and `<EMAIL>` would be translated or broken. They are swapped for unique 4-digit codes (NLLB copies numbers), then put back.
* **Automatic flags.** A row is flagged if the output is empty, a placeholder was lost, it is in the wrong script, its length is odd, or words repeat. Flagged rows are **not used** unless you mark them Good or Understandable.

In [ ]:
import re
from collections import Counter

SINHALA_RE = re.compile(r"[඀-෿]")
PLACEHOLDER_RE = re.compile(r"<NUMBER>|<EMAIL>")
SENT_SPLIT_RE = re.compile(r"(?<=[.!?෴।])\s+")


def protect(text):
    # replace placeholders with codes such as 7301 / 8301; returns (text, {code: placeholder})
    mapping, n = {}, {"<NUMBER>": 7300, "<EMAIL>": 8300}

    def swap(m):
        ph = m.group(0)
        n[ph] += 1
        while str(n[ph]) in text:          # never reuse a number that is already in the text
            n[ph] += 1
        mapping[str(n[ph])] = ph
        return str(n[ph])

    return PLACEHOLDER_RE.sub(swap, text), mapping


def restore(text, mapping):
    lost = [code for code in mapping if code not in text]
    for code, ph in mapping.items():
        text = text.replace(code, ph)
    return text, lost


def sentences(text):
    out = []
    for s in SENT_SPLIT_RE.split(text):
        words = s.split()
        if len(words) <= MAX_SENTENCE_WORDS:
            out.append(s)
            continue
        piece = []                          # very long sentence: cut after commas
        for w in words:
            piece.append(w)
            if len(piece) >= MAX_SENTENCE_WORDS // 2 and w.endswith((",", ";")):
                out.append(" ".join(piece)); piece = []
        if piece:
            out.append(" ".join(piece))
    return [s for s in out if s.strip()]


def sinhala_share(text):
    letters = [c for c in text if c.isalpha() or SINHALA_RE.match(c)]
    return sum(bool(SINHALA_RE.match(c)) for c in letters) / max(len(letters), 1)


def quality_flags(original, translated, target, lost):
    flags = []
    if not translated.strip():
        return ["empty"]
    if lost:
        flags.append("placeholder lost")
    share = sinhala_share(translated)
    if (target == "si" and share < 0.5) or (target == "en" and share > 0.2):
        flags.append("wrong script")
    ratio = len(translated) / max(len(original), 1)
    if not 0.35 <= ratio <= 3.0:
        flags.append(f"length ratio {ratio:.2f}")
    words = translated.split()
    tri = Counter(zip(words, words[1:], words[2:]))
    if (tri and max(tri.values()) >= 3) or any(
            words[i] == words[i + 1] == words[i + 2] == words[i + 3] for i in range(len(words) - 3)):
        flags.append("repetition")
    return flags


# quick self-check of the helpers (invented text)
t, m = protect("Call me on <NUMBER> or <EMAIL>. My bill is wrong.")
assert t == "Call me on 7301 or 8301. My bill is wrong." and restore(t, m)[0].count("<") == 2
assert sentences("One. Two! Three") == ["One.", "Two!", "Three"]
print("helpers OK")

## Part 4: Load NLLB-200 and test it on two invented sentences

In [ ]:
import time
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

assert torch.cuda.is_available(), "No GPU. Settings, then Accelerator, then GPU T4 x1."
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to("cuda").eval()
tokenizers = {}


def get_tokenizer(src_code):
    if src_code not in tokenizers:
        tokenizers[src_code] = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=src_code)
    return tokenizers[src_code]


def translate(texts, src_code, tgt_code, batch_size=BATCH_SIZE):
    # translates a list of sentences; identical sentences are translated once
    tok = get_tokenizer(src_code)
    tgt_id = tok.convert_tokens_to_ids(tgt_code)
    unique = sorted(set(texts), key=len)            # similar lengths together = faster
    result = {}
    for i in range(0, len(unique), batch_size):
        batch = unique[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=400).to("cuda")
        max_new = min(512, int(enc["input_ids"].shape[1] * 3) + 10)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, num_beams=NUM_BEAMS,
                                 max_new_tokens=max_new, no_repeat_ngram_size=4)
        for s, out in zip(batch, tok.batch_decode(gen, skip_special_tokens=True)):
            result[s] = out.strip()
        if (i // batch_size) % 20 == 0:
            print(f"  {min(i + batch_size, len(unique))}/{len(unique)} sentences", flush=True)
    return [result[t] for t in texts]


# invented examples - check the output looks sensible before the long run
print(translate(["My electricity bill for this month is much higher than usual."], "eng_Latn", "sin_Sinh"))
print(translate(["අපේ පාරේ කාණු අවහිර වෙලා වතුර පාරට එනවා."], "sin_Sinh", "eng_Latn"))

## Part 5: Translate
Results are saved after each direction (`translations_partial.xlsx`), so a stopped session does not lose finished work.

In [ ]:
results = []
for name, rows, src_code, tgt_code, tgt_lang in jobs:
    start = time.time()
    prepared = []                                    # (row, [sentences], mapping)
    for r in rows.itertuples():
        protected, mapping = protect(r.text)
        prepared.append((r, sentences(protected), mapping))
    all_sents = [s for _, sents, _ in prepared for s in sents]
    print(f"{name}: {len(rows)} complaints, {len(all_sents)} sentences ({len(set(all_sents))} unique)")
    translated = iter(translate(all_sents, src_code, tgt_code))

    for r, sents, mapping in prepared:
        out = " ".join(next(translated) for _ in sents)
        out, lost = restore(out, mapping)
        flags = quality_flags(r.text, out, tgt_lang, lost)
        results.append({
            "source_id": r.source_id, "institution": r.institution, "department": r.department,
            "split": r.split, "direction": name, "source_language": r.language,
            "target_language": tgt_lang, "original_text": r.text, "translated_text": out,
            "auto_flag": "; ".join(flags), "in_review_sample": False, "review": "",
            "reviewer_note": "",
        })
    pd.DataFrame(results).to_excel(OUT / "translations_partial.xlsx", index=False)
    print(f"  done in {(time.time() - start) / 60:.1f} min\n")

## Part 6: Write `translations.xlsx`
* `review` (drop-down): **Good** / **Understandable** / **Wrong**. Fill it for every row where `in_review_sample` is TRUE (50 per direction), and for any flagged row you want to keep.
* You may also **correct `translated_text`** directly.
* `build_dataset_v2.py` uses a row if `review` is not Wrong **and** (`auto_flag` is empty **or** `review` is Good/Understandable).

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation

res = pd.DataFrame(results)
for name in res["direction"].unique():
    ok = res[(res["direction"] == name) & (res["auto_flag"] == "")]
    pick = ok.sample(min(REVIEW_SAMPLE_PER_DIRECTION, len(ok)), random_state=SEED).index
    res.loc[pick, "in_review_sample"] = True
res = res.sort_values(["in_review_sample", "direction"], ascending=[False, True])

summary = (res.assign(flagged=res["auto_flag"] != "")
              .groupby(["direction", "department", "split"])
              .agg(translated=("source_id", "size"), auto_flagged=("flagged", "sum"))
              .reset_index())
flag_counts = (res["auto_flag"].str.split("; ").explode().replace("", pd.NA).dropna()
               .str.replace(r" \d.*", "", regex=True).value_counts()
               .rename_axis("flag").reset_index(name="rows"))

path = OUT / "translations.xlsx"
with pd.ExcelWriter(path, engine="openpyxl") as xl:
    res.to_excel(xl, sheet_name="translations", index=False)
    summary.to_excel(xl, sheet_name="summary", index=False)
    flag_counts.to_excel(xl, sheet_name="flags", index=False)
    pd.DataFrame({"setting": ["model", "LECO share", "MC share", "splits", "beams", "seed"],
                  "value": [MODEL_NAME, LECO_TO_SINHALA_SHARE, MC_TO_ENGLISH_SHARE,
                            ", ".join(SPLITS_TO_TRANSLATE), NUM_BEAMS, SEED]}
                 ).to_excel(xl, sheet_name="settings", index=False)

wb = load_workbook(path)
ws = wb["translations"]
cols = {c.value: c.column_letter for c in ws[1]}
dv = DataValidation(type="list", formula1='"Good,Understandable,Wrong"', allow_blank=True)
ws.add_data_validation(dv)
dv.add(f"{cols['review']}2:{cols['review']}{ws.max_row}")
for name, width in [("original_text", 60), ("translated_text", 60), ("auto_flag", 22),
                    ("review", 16), ("reviewer_note", 30), ("direction", 13), ("department", 18)]:
    ws.column_dimensions[cols[name]].width = width
for row in ws.iter_rows(min_row=2):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical="top")
for c in ws[1]:
    c.font = Font(bold=True)
ws.freeze_panes = "A2"
wb.save(path)

print("Saved", path)
print(summary.groupby("direction")[["translated", "auto_flagged"]].sum().to_string())
print("\nAutomatic flags:\n" + (flag_counts.to_string(index=False) if len(flag_counts) else "none"))

## Next
1. Save Version (Save & Run All). When it finishes, open **Output** and download `translations.xlsx`.
2. On your computer, review the rows with `in_review_sample = TRUE` and fill the `review` column.
3. Put `translations.xlsx` next to `build_dataset_v2.py` and run it. It writes `dataset_v2/`.
4. Report in the methodology: model, shares, number translated, automatic flags, and the review percentages (Good / Understandable / Wrong).